# 🔥 Chapter 16: DevOps War Stories and Interviews

📖 Book: Chapter 16, pp. 661–710 · ⏱️ 2 hours · 🧱 Needs: Ch 1–15 · Level: All Levels

---

> **🎯 Key idea:** Automation beats hierarchy: blameless postmortems, profiling before optimizing (`py-spy`/`cProfile`/`strace`), and safe validation without `assert` prevent outages.

> **💡 Tip:** Run cells sequentially top-to-bottom (`Shift + Enter`).

> **⚠️ Warning:** Cells tagged `external` may require extra packages; cloud stacks incur costs if deployed.

## 🔄 Then vs. Now

> **📜 Then (book, ~2020):** `cProfile` + `strace` + manual APM  
> **🆕 Now (2026):** `py-spy`, `scalene`, OpenTelemetry tracing  
> **Why:** Zero-overhead live sampling & trace correlation

> **📜 Then (book, ~2020):** `assert` in app code (antipattern)  
> **🆕 Now (2026):** Explicit `if ...: raise ValueError(...)`  
> **Why:** `python3 -O` strips all `assert`s

---

## 4. 💡 Conceptual Cards

### 🚨 Real-World Problem
A CLI tool suddenly takes 60 seconds just to start, or an RDS database hits 100% CPU every evening—and guessing the cause wastes hours. Without blameless postmortems and profiling, teams repeat the same outages. This final chapter turns real war stories into profiling, caching, and code-hygiene habits.

### Concept: Blameless Postmortems
**What:** Timeline -> Root Cause -> Fix -> Prevention  
**Why:** Turns outages into automated CI/monitoring guardrails  
**How:** Document what failed & automate the check  
**When:** After every production incident

📖 *Book listing:* 60-Second Python `strace` Diagnosis (pp. 669–670)

**Plain-language analogy:** A blameless postmortem asks 'what guardrail was missing?' instead of 'who clicked the button?'—just as aviation turns crashes into checklists.

**Walkthrough:** The book profiles a 60-second Python CLI startup with `strace`, uncovering >100,000 `stat` syscalls over a shared NFS filer during module imports.

### Concept: Profiling & Read-Through Caching
**What:** `cProfile`/`py-spy` + `TTLCache`  
**Why:** 90% of DB meltdowns are repeated read queries  
**How:** Measure hotspots first; cache read routes with TTL  
**When:** Before optimizing or scaling DBs

📖 *Book listing:* RDS SQL Server Meltdown & Nginx Caching Rescue (pp. 670–672)

**Plain-language analogy:** Read-through caching is like keeping today's weather forecast taped to the front desk so 1,000 callers don't force you to walk outside every time.

**Walkthrough:** The book traces peak DB CPU to repetitive read-only routes via APM and cuts load immediately by adding short-TTL read-through caching.

## 6. 🟢 Beginner Examples

#### Example 1: Beginner Example 1: Safe runtime validation (never assert in prod!)
*Expected output:* Printed labelled output below.

In [1]:
# Beginner Example 1: Safe runtime validation (never assert in prod!)
def validate_port(port: int) -> int:
    if not isinstance(port, int) or isinstance(port, bool) or not (1 <= port <= 65535):
        raise ValueError(f'Invalid port: {port!r}')
    return port

print('Valid port:', validate_port(8080))

Valid port: 8080


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 2: Beginner Example 2: High-resolution latency measurement
*Expected output:* Printed labelled output below.

In [2]:
# Beginner Example 2: High-resolution latency measurement
import time

def measure_ms(fn, *args):
    t0 = time.perf_counter()
    res = fn(*args)
    return res, round((time.perf_counter() - t0) * 1000, 3)

print('Sum latency (ms):', measure_ms(sum, range(10000)))

Sum latency (ms): (49995000, 0.13)


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 3: Beginner Example 3: Bounded lru_cache for deterministic lookups
*Expected output:* Printed labelled output below.

In [3]:
# Beginner Example 3: Bounded lru_cache for deterministic lookups
import functools

@functools.lru_cache(maxsize=32)
def normalize_env(name: str) -> str:
    return name.strip().upper()

normalize_env('prod'); normalize_env('prod')
print('Cache info:', normalize_env.cache_info())

Cache info: CacheInfo(hits=1, misses=1, maxsize=32, currsize=1)


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

## 7. 🟡 Intermediate Examples: `cProfile` Bottleneck Finder & `TTLCache` Rescue

#### Example 4: Intermediate Example 1: cProfile bottleneck detector (and py-spy note)
*Expected output:* Printed labelled output below.

In [4]:
# Intermediate Example 1: cProfile bottleneck detector (and py-spy note)
import cProfile, io, pstats

def slow_join(n: int) -> str:
    return ''.join(str(i) for i in range(n))

prof = cProfile.Profile()
prof.enable(); slow_join(5000); prof.disable()
buf = io.StringIO()
pstats.Stats(prof, stream=buf).sort_stats('tottime').print_stats(4)
print('\n'.join(buf.getvalue().splitlines()[:7]))

         5045 function calls in 0.002 seconds

   Ordered by: internal time
   List reduced from 22 to 4 due to restriction <4>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     5001    0.001    0.000    0.001    0.000 /tmp/ipykernel_342997/2757059428.py:5(<genexpr>)


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 5: Intermediate Example 2: Read-through TTLCache (Incident 4 RDS rescue pattern)
*Expected output:* Printed labelled output below.

In [5]:
# Intermediate Example 2: Read-through TTLCache (Incident 4 RDS rescue pattern)
import time

class TTLCache:
    def __init__(self, ttl_seconds: float = 5.0) -> None:
        self.ttl = ttl_seconds
        self._store: dict[str, tuple[float, object]] = {}
        self.db_hits = 0

    def get_or_fetch(self, key: str, loader):
        now = time.monotonic()
        if key in self._store and now < self._store[key][0]:
            return self._store[key][1]
        self.db_hits += 1
        val = loader(key)
        self._store[key] = (now + self.ttl, val)
        return val

cache = TTLCache(ttl_seconds=10.0)
for _ in range(5):
    cache.get_or_fetch('scores', lambda k: {'game': 'A vs B'})
print('5 requests -> DB hits:', cache.db_hits)

5 requests -> DB hits: 1


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

## 8. 🚀 Real-World DevOps Applications (Personal DevOps Learning Track)
*Supplementary extension:* **AST Production Code Hygiene Auditor** flagging `assert`, bare `except:`, and `print()` calls.

#### Example 6: import ast
*Expected output:* Printed labelled output below.

In [6]:
import ast

def audit_prod_code(source: str) -> list[str]:
    findings: list[str] = []
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, ast.Assert):
            findings.append(f'Line {node.lineno}: avoid assert in prod')
        elif isinstance(node, ast.ExceptHandler) and node.type is None:
            findings.append(f'Line {node.lineno}: avoid bare except:')
        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == 'print':
            findings.append(f'Line {node.lineno}: prefer logging over print()')
    return sorted(findings)

print('Findings:', audit_prod_code('def f(x):\n    assert x > 0\n    try:\n        print(x)\n    except:\n        pass\n'))

Findings: ['Line 2: avoid assert in prod', 'Line 4: prefer logging over print()', 'Line 5: avoid bare except:']


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 7: Real Tool Demo: strace syscall counter on Python import (guarded with shutil.which)
*Expected output:* Printed labelled output below.

In [7]:
# Real Tool Demo: strace syscall counter on Python import (guarded with shutil.which)
import shutil, subprocess
if shutil.which('strace'):
    cp = subprocess.run(['strace', '-c', '-e', 'openat', 'python3', '-c', 'import json'], capture_output=True, text=True)
    print('\n'.join(cp.stderr.splitlines()[:6]))
else:
    print('strace not installed; skipped')

% time     seconds  usecs/call     calls    errors syscall
------ ----------- ----------- --------- --------- ----------------
100.00    0.000165           2        77        13 openat
------ ----------- ----------- --------- --------- ----------------
100.00    0.000165           2        77        13 total


**What the output means:** Checks whether the real CLI tool exists on `PATH`; if found, runs a safe read-only/dry-run check, otherwise skips cleanly.

> **🧪 Try it:** Try adding `--help` or inspecting `cp.returncode` to see how exit codes behave.

---

## 9. ⚠️ Common Mistakes & Troubleshooting

- ❌ **Using `assert` for prod checks** -> Disabled silently by `python3 -O`. ✅ Raise `ValueError` explicitly.
- ❌ **Guessing bottlenecks** -> Wastes time optimizing wrong code. ✅ Measure with `py-spy`/`cProfile`/`strace`.
- ❌ **Unbounded caches** -> Slow memory leak -> OOM. ✅ Always set TTL or `maxsize`.

> **🧪 Try it:** Run the starter cell below—it prints `TODO` until you implement the body.

#### 🧪 Starter Exercise: `Starter Exercise Cell (implement validate_token and run to verify!)`
*Expected output:* `TODO` message until implemented, then `✅ Starter checks PASSED!`.

In [8]:
# Starter Exercise Cell (implement validate_token and run to verify!)
def _run_starter_checks():
    assert validate_token('a' * 16) == 'a' * 16
def validate_token(token: str) -> str:
    raise NotImplementedError('Implement validate_token')

try:
    _run_starter_checks()
    print('✅ Starter checks PASSED!')
except NotImplementedError as e:
    print(f'🧪 TODO: {e}')

🧪 TODO: Implement validate_token


## 12. 🧠 Knowledge Checks (Inline Q&A for Practice)
**Q1:** Why avoid `assert` for runtime validation in production?

<details>
<summary>✅ Answer 1</summary>

`python3 -O` strips `assert` statements, and unhandled `AssertionError`s crash services.

</details>

**Q2:** What did `strace` reveal in the 60-second Python script war story?

<details>
<summary>✅ Answer 2</summary>

>100,000 `stat` syscalls over a slow shared NFS filer during module imports.

</details>

**Q3:** How did caching rescue the overloaded RDS SQL Server?

<details>
<summary>✅ Answer 3</summary>

Offloaded repetitive read-only queries to Nginx caching, slashing DB CPU load.

</details>

**Q4:** What is the HIPO antipattern?

<details>
<summary>✅ Answer 4</summary>

Highest Paid Person's Opinion—making technical decisions by authority instead of data.

</details>

**Q5:** What modern profilers complement `cProfile` today?

<details>
<summary>✅ Answer 5</summary>

`py-spy` (live sampling), `scalene` (CPU+memory), and OpenTelemetry tracing.

</details>

## 13. 🔑 Hints & Complete Solutions

<details>
<summary>💭 Hints (click to expand)</summary>

- **Ex 5**: Inspect `ast.Call.keywords` for `kw.arg == 'shell'` and `isinstance(kw.value, ast.Constant) and kw.value.value is True`.

</details>

> **💡 Tip:** Complete solutions are in the collapsed/marked cell below.

<details>
<summary>✅ Complete Solutions (expand/run cell below to verify)</summary>

Run the verified solutions cell below to check all exercise & challenge assertions.

</details>

In [9]:
# Complete Solutions + Assert Verification
from dataclasses import dataclass

def validate_token(token: str) -> str:
    if len(token) < 16:
        raise ValueError('Token must be >= 16 chars')
    return token

@dataclass
class Postmortem:
    title: str
    root_cause: str
    fix: str
    prevention: str

def render_postmortem_md(pm: Postmortem) -> str:
    for f in (pm.title, pm.root_cause, pm.fix, pm.prevention):
        if not f.strip():
            raise ValueError('All postmortem fields must be non-empty')
    return f'# {pm.title}\n- **Root Cause:** {pm.root_cause}\n- **Fix:** {pm.fix}\n- **Prevention:** {pm.prevention}'

pm = Postmortem('RDS Spike', 'Uncached queries', 'Added Nginx TTL cache', 'Load test + APM')
assert '# RDS Spike' in render_postmortem_md(pm)
assert validate_token('a' * 16) == 'a' * 16
print('All Chapter 16 assertions PASSED!')

All Chapter 16 assertions PASSED!


---

## 14. 📌 Chapter Summary & Quick Reference

- `time.perf_counter()` — High-res latency timing (e.g. `t1 - t0`)
- `strace -c -e openat cmd` — Count OS syscalls (e.g. Diagnose slow NFS imports)
- `ast.walk(ast.parse(src))` — Static hygiene audit (e.g. Flag `assert` & bare `except:`)

### ✅ What You Can Now Do
- Analyze outages using blameless postmortems (Timeline -> Root Cause -> Fix -> Prevention).
- Profile latency and bottlenecks with `time.perf_counter()`, `cProfile`, `strace`, `py-spy`, and `scalene`.
- Audit production Python code with `ast.walk()` to eliminate `assert`, bare `except:`, and `print()`.

### ➡️ Next Chapter
Next, head back to the root `README.md` to build the Final Capstone Project combining CLI, logging, `pytest`, Docker, and Kubernetes!